# Text Completion Fine-Tuning with GPT-2

Fine-tuning GPT-2 for text completion on a curated set of quotations. GPT-2 is a decoder-only (autoregressive) transformer - it predicts the next token given all preceding tokens. This makes it a natural fit for completion tasks: given the first half of a quote, generate the rest.

Each example is a (context, target) pair formatted with separator tokens so the model learns to transition from the prompt into the completion. We fine-tune the full model on this small dataset, which teaches GPT-2 the style and content of these specific quotations while retaining its general language ability.

In [3]:
dataset = [
    # On power and politics
    ("He who fights with monsters should look to it", "that he himself does not become a monster."),
    ("If you gaze long into an abyss,", "the abyss also gazes into you."),
    ("Whoever fights monsters should see to it", "that in the process he does not become one."),
    ("The strong do what they can", "and the weak suffer what they must."),
    ("Laws are spider webs through which", "the big flies pass and the little ones get caught."),
    ("Politics is the art of looking for trouble, finding it everywhere,", "diagnosing it incorrectly, and applying the wrong remedies."),
    ("Power tends to corrupt,", "and absolute power corrupts absolutely."),
    ("It is dangerous to be right", "in matters where established men are wrong."),

    # On war
    ("In war,", "truth is the first casualty."),
    ("War is the continuation of politics", "by other means."),
    ("All warfare is based", "on deception."),
    ("The supreme art of war", "is to subdue the enemy without fighting."),
    ("To know your enemy,", "you must become your enemy."),
    ("Older men declare war.", "But it is youth that must fight and die."),
    ("There never was a good war", "or a bad peace."),

    # On human nature
    ("Hell", "is other people."),
    ("Man is born free,", "and everywhere he is in chains."),
    ("We are all in the gutter,", "but some of us are looking at the stars."),
    ("No man is a hero", "to his valet."),
    ("Hypocrisy is the tribute", "that vice pays to virtue."),
    ("We all have strength enough", "to bear the misfortunes of others."),
    ("The heart has its reasons", "of which reason knows nothing."),
    ("Men are born ignorant, not stupid.", "They are made stupid by education."),

    # On fate, death, suffering
    ("Call no man happy", "until he is dead."),
    ("Death is the solution to all problems.", "No man, no problem."),
    ("A single death is a tragedy;", "a million deaths is a statistic."),
    ("Life is a long lesson", "in humility."),
    ("To live is to suffer,", "to survive is to find some meaning in the suffering."),
    ("That which does not kill us", "makes us stronger."),
    ("We are all condemned to death,", "only the date of execution is uncertain."),
    ("The world breaks everyone,", "and afterward many are strong at the broken places."),

    # On truth, lies, and illusion
    ("There are no facts,", "only interpretations."),
    ("History is a set of lies", "agreed upon."),
    ("The truth is rarely pure", "and never simple."),
    ("A lie told often enough", "becomes the truth."),
    ("Believe nothing you hear,", "and only one half that you see."),
    ("Man is least himself when he talks in his own person.", "Give him a mask, and he will tell you the truth."),

    # On justice and morality
    ("Justice is the means by which", "established injustices are sanctioned."),
    ("The law, in its majestic equality,", "forbids rich and poor alike to sleep under bridges."),
    ("Every man is guilty", "of all the good he did not do."),
    ("Conscience is the inner voice", "that warns us somebody may be looking."),
    ("The greatest crimes in the world", "are not committed by people breaking the rules."),

    # On hope, faith, and despair
    ("Hope is the worst of evils,", "for it prolongs the torment of man."),
    ("Where there is no hope,", "it is incumbent on us to invent it."),
    ("Man is condemned", "to be free."),
    ("It is not death that a man should fear,", "but he should fear never beginning to live."),

    # Older / proverbial
    ("Even the gods", "cannot strive against necessity."),
    ("The arrow that has left the bow", "never returns."),
    ("When elephants fight,", "it is the grass that suffers."),
    ("Pray to God,", "but row away from the rocks."),
]

In [6]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
formatted_data = [(f"[CLS] {context} [SEP] {target} [SEP]",) for context, target in dataset]
numerical_data = [tokenizer.encode(example[0], add_special_tokens=True) for example in formatted_data]

## Tokenization

GPT-2 uses byte-pair encoding (BPE) and doesn't ship with a pad token - it was trained on continuous text without padding. We assign the end-of-sequence token (`eos_token`) as the pad token, which is the standard convention since it already signals "no more meaningful content."

Each example is formatted as `[CLS] context [SEP] target [SEP]` and encoded into token IDs.

In [7]:
import torch

max_length = max(len(seq) for seq in numerical_data)
padded_data = [seq + [tokenizer.pad_token_id] * (max_length - len(seq)) for seq in numerical_data]

input_ids = torch.tensor(padded_data)

## Padding

Sequences have different lengths but need to be stacked into a single tensor for batching. We right-pad each sequence to the length of the longest one using the pad token ID.

In [ ]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
  def __init__(self, input_ids):
    self.input_ids = input_ids

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return {'input_ids': self.input_ids[idx]}

## Dataset and DataLoader

Standard PyTorch data pipeline: a `Dataset` that serves individual examples by index, and a `DataLoader` that handles batching and shuffling.

In [ ]:
from torch.utils.data import DataLoader

batch_size = 4
custom_dataset = CustomDataset(input_ids)
dataloader = DataLoader(custom_dataset, batch_size=batch_size, shuffle=True)

In [10]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained('gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11703.12it/s]


## Model

Loading the pre-trained GPT-2 language model head. `GPT2LMHeadModel` includes the transformer decoder plus the language modelling head that projects hidden states back into the vocabulary space to predict the next token.